# 05b — Clustering HDBSCAN

No requiere K. Variando min_cluster_size = [100,500,1000,2000].

In [1]:
import pandas as pd
import numpy as np
import time, joblib
from pathlib import Path
from datetime import datetime
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
DATA_MODELS = ROOT / 'data' / 'data_models_gustos'
INFORMES_BASE = ROOT / 'informes' / 'fase4_gustos' / '04_clustering'
DATA_MODELS.mkdir(parents=True, exist_ok=True)

MASTER_VERSIONS = {
    'v1_conservative': DATA_QC / 'master_table_gustos_v1_conservative.parquet',
    'v2_intermediate': DATA_QC / 'master_table_gustos_v2_intermediate.parquet',
    'v3_aggressive':   DATA_QC / 'master_table_gustos_v3_aggressive.parquet',
}

def reduce_high_card(df, max_unique=10):
    out = df.copy()
    for cat in out.select_dtypes(include=['object', 'category']).columns.tolist():
        n = out[cat].nunique(dropna=True)
        if n > max_unique:
            top = out[cat].value_counts().head(max_unique).index.tolist()
            out[cat] = out[cat].where(out[cat].isin(top), 'OTHER')
    return out

def preprocess_master(master_df, nan_threshold_zero=0.30):
    """Returns (X_scaled, preprocessor, feature_names_out)."""
    df = master_df.drop(columns=['user_id']).copy()
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
    nan_pcts = df[numeric_cols].isna().mean()
    num_low = nan_pcts[nan_pcts < nan_threshold_zero].index.tolist()
    num_high = nan_pcts[nan_pcts >= nan_threshold_zero].index.tolist()
    print(f'  num_low_nan: {len(num_low)} | num_high_nan: {len(num_high)} | cat: {len(categorical_cols)}')
    df = reduce_high_card(df, max_unique=10)
    preproc = ColumnTransformer([
        ('num_low', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())]), num_low),
        ('num_high', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value=0)), ('sc', RobustScaler())]), num_high),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_cols),
    ])
    X = preproc.fit_transform(df)
    feature_names = preproc.get_feature_names_out()
    return X, preproc, feature_names

def load_and_preprocess(version_name):
    master = pd.read_parquet(MASTER_VERSIONS[version_name])
    print(f'[{version_name}] shape={master.shape}')
    user_ids = master['user_id'].values
    X, preproc, names = preprocess_master(master)
    return X, user_ids, preproc, names, master.shape

import hdbscan
from sklearn.metrics import silhouette_score, davies_bouldin_score

REPORT = INFORMES_BASE / '05b_hdbscan'
REPORT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
MIN_SIZES = [100, 500, 1000, 2000]
SAMPLE_TRAIN = 50_000
SAMPLE_SIL = 20_000
results = []


In [2]:

for version in MASTER_VERSIONS:
    print(f'\n=== {version} ===')
    X, user_ids, preproc, names, shape = load_and_preprocess(version)
    print(f'  X shape: {X.shape}')

    rng = np.random.RandomState(RANDOM_STATE)
    if len(X) > SAMPLE_TRAIN:
        sample_idx = rng.choice(len(X), SAMPLE_TRAIN, replace=False)
        X_train = X[sample_idx]
        print(f'  Train sample: {len(X_train):,}')
    else:
        sample_idx = np.arange(len(X))
        X_train = X

    for min_size in MIN_SIZES:
        t0 = time.time()
        clusterer = hdbscan.HDBSCAN(min_cluster_size=min_size, metric='euclidean', core_dist_n_jobs=-1, prediction_data=True)
        clusterer.fit(X_train)
        labels_train = clusterer.labels_
        # Proyectar al full
        labels_all, strengths = hdbscan.approximate_predict(clusterer, X)
        n_clusters = int(len(set(labels_all)) - (1 if -1 in labels_all else 0))
        n_outliers = int((labels_all == -1).sum())
        outlier_pct = n_outliers / len(labels_all)
        # silhouette sobre subsample sin outliers
        non_out = labels_all != -1
        if non_out.sum() > 100 and len(set(labels_all[non_out])) > 1:
            sil_idx = rng.choice(np.where(non_out)[0], min(SAMPLE_SIL, int(non_out.sum())), replace=False)
            try:
                sil = float(silhouette_score(X[sil_idx], labels_all[sil_idx]))
            except Exception:
                sil = float('nan')
        else:
            sil = float('nan')
        try:
            db = float(davies_bouldin_score(X[non_out], labels_all[non_out])) if non_out.sum() > 100 and len(set(labels_all[non_out])) > 1 else float('nan')
        except Exception:
            db = float('nan')
        elapsed = time.time() - t0
        results.append({
            'algorithm': 'HDBSCAN', 'version': version, 'min_cluster_size': min_size,
            'n_clusters_actual': n_clusters, 'n_outliers': n_outliers, 'outlier_pct': outlier_pct,
            'silhouette': sil, 'davies_bouldin': db, 'elapsed_s': elapsed,
        })
        print(f'  min_size={min_size}: clusters={n_clusters} outliers={outlier_pct:.1%} sil={sil:.4f} t={elapsed:.0f}s')
        joblib.dump({'model': clusterer, 'labels': labels_all, 'user_ids': user_ids, 'sample_idx': sample_idx}, DATA_MODELS / f'hdbscan_{version}_min{min_size}.joblib')

res_df = pd.DataFrame(results)
res_df.to_csv(REPORT / 'metrics.csv', index=False)
print(f'\n{res_df.to_string(index=False)}')

best = res_df.dropna(subset=['silhouette']).sort_values('silhouette', ascending=False).head(3)
lines = [f'# 05b HDBSCAN', f'**Fecha**: {datetime.now().strftime("%Y-%m-%d %H:%M")}', '',
         '## Top 3 configuraciones', '', '| Versión | min_size | n_clusters | outlier% | silhouette | DB |', '|---|---:|---:|---:|---:|---:|']
for _, r in best.iterrows():
    lines.append(f"| {r['version']} | {int(r['min_cluster_size'])} | {int(r['n_clusters_actual'])} | {r['outlier_pct']:.1%} | {r['silhouette']:.4f} | {r['davies_bouldin']:.4f} |")
(REPORT / 'execution_report.md').write_text('\n'.join(lines))



=== v1_conservative ===
[v1_conservative] shape=(114412, 115)


/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_1989/2709075291.py:36: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_1989/2709075291.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence t

  num_low_nan: 86 | num_high_nan: 24 | cat: 4


  X shape: (114412, 132)
  Train sample: 50,000


  min_size=100: clusters=8 outliers=15.0% sil=0.7977 t=489s


...

  min_size=500: clusters=4 outliers=17.2% sil=0.8609 t=446s


  min_size=1000: clusters=4 outliers=19.1% sil=0.8770 t=503s


  min_size=2000: clusters=0 outliers=100.0% sil=nan t=264s

algorithm         version  min_cluster_size  n_clusters_actual  n_outliers  outlier_pct  silhouette  davies_bouldin  elapsed_s
  HDBSCAN v1_conservative               100                  8       17207     0.150395    0.797651        0.364763 489.033910
  HDBSCAN v1_conservative               500                  4       19884     0.173793    0.857413        0.340461 535.392436
  HDBSCAN v1_conservative              1000                  4       22089     0.193065    0.872629        0.284241 653.673043
  HDBSCAN v1_conservative              2000                  0      114412     1.000000         NaN             NaN 317.350264
  HDBSCAN v2_intermediate               100                  8       17231     0.150605    0.799641        0.363355 414.769791
  HDBSCAN v2_intermediate               500                  4       19824     0.173269    0.858483        0.339815 480.229748
  HDBSCAN v2_intermediate              1000        

/Users/jezquerro/Documents/tfg/.venv/lib/python3.14/site-packages/hdbscan/prediction.py:391: UserWarning: Clusterer does not have any defined clusters, new data will be automatically predicted as noise.
  warn('Clusterer does not have any defined clusters, new data'


333